# Django, Views

## Introduction to Django Views
---

In this lesson, you'll learn how to write **Django views** - the heart of your application's logic.

Views are Python functions or classes that receive HTTP requests and return HTTP responses.

1. [Function-Based Views (FBV)](#Function-Based-Views-(FBV)),
    - [Basic Structure](#Basic-Structure),
    - [Returning Different Response Types](#Returning-Different-Response-Types),
2. [Request and Response Objects](#Request-and-Response-Objects),
    - [The Request Object](#The-Request-Object),
    - [The Response Object](#The-Response-Object),
3. [Class-Based Views (CBV)](#Class-Based-Views-(CBV)),
    - [Basic CBV Structure](#Basic-CBV-Structure),
    - [HTTP Method Handlers](#HTTP-Method-Handlers),
4. [FBV vs CBV - When to Use Which](#FBV-vs-CBV---When-to-Use-Which),
    - [Comparison](#Comparison),
    - [Practical Guidelines](#Practical-Guidelines),
5. [🧠 EXERCISE 🧠, Create Views for Bookstore](#🧠-EXERCISE-🧠,-Create-Views-for-Bookstore).

<br>

## Function-Based Views (FBV)

---

### Basic Structure

---

A **function-based view** (FBV) is simply a Python function that:

1. Takes an `HttpRequest` object as the first argument
2. Returns an `HttpResponse` object

<br>

```
Request  →  View Function  →  Response
```

In [ ]:
# Example: Simplest possible view

from django.http import HttpRequest, JsonResponse
from blog.models import Blog


def blog_list(request: HttpRequest):
    blog_ids = [blog.id for blog in Blog.objects.all()]
    return JsonResponse(blog_ids, safe=False)

We've already tested the outcome from the table `blog_blog`.
```bash
curl "http://localhost:8000/blog"
# Result: [1]
```

In [ ]:
# Example: View with URL parameter, 'id'

from http import HTTPStatus
from django.http import JsonResponse

def blog_detail(request: HttpRequest, id: int):
    try:
        result = Blog.objects.get(id=id)
    
    except Blog.DoesNotExist:
        return JsonResponse(
            {'error': 'Blog not found'},
            status=HTTPStatus.NOT_FOUND,
        )
    return JsonResponse({
        'id': result.id,
        'title': result.title,
        'author': result.author,
        'published_date': result.published_date.isoformat()
    })

# URL pattern: path('blogs/<int:pk>/', views.blog_detail)
# Visit: /blogs/1/  →  "{"id": 1, "title": "My First Blog", "author": "John Doe", "published_date": "2021-01-01"}"

But you can use even more parameters.

<br>

### Returning Different Response Types

---

Views can return different types of responses:

In [ ]:
# Example: Different response types using the same view name
# Variant 1: Plain text response

from django.http import HttpResponse
from django.shortcuts import get_object_or_404
from models import Blog


def blog_detail(request: HttpRequest, id: int) -> HttpResponse:
    blog = get_object_or_404(Blog, id=id)
    return HttpResponse(
        f"Blog #{blog.id}: {blog.title} by {blog.author}",
        content_type="text/plain",
    )


# ❯ curl "http://localhost:8000/blog/1/"
# Blog #1: Django Bootcamp by Matous Holinka%   

🔑 Article listing, profile details, contact form (everything that goes through `render(<request>, 'my-template.html')`).

In [ ]:
# Variant 2: HTML response

from django.http import HttpResponse
from django.shortcuts import get_object_or_404
from blog.models import Blog


def blog_detail(request, id):
    blog = get_object_or_404(Blog, id=id)
    return HttpResponse(
        f"<h1>{blog.title}</h1><p>Author: {blog.author}</p>",
        content_type="text/html",
    )

# ❯ curl "http://localhost:8000/blog/1/"
# <h1>Django Bootcamp</h1><p>Author: Matous Holinka</p>

🔑 Its an output for technical tools, dynamically generated `robots.txt`, or a simple "OK" for some monitoring script (Health Check).

In [ ]:
# Variant 3: JSON response (API style)

from django.http import JsonResponse
from django.shortcuts import get_object_or_404
from blog.models import Blog


def blog_detail(request, id):
    blog = get_object_or_404(Blog, id=id)
    return JsonResponse(
        {
            "id": blog.id,
            "title": blog.title,
            "author": blog.author,
            "published_date": blog.published_date.isoformat(),
        }
    )


# ❯ curl "http://localhost:8000/blog/1/"
# {"id": 1, "title": "Django Bootcamp", "author": "Matous Holinka", "published_date": "2026-03-26"}%  

🔑 Whisperer in search, loading more posts without refreshing the page, API interface.

In [ ]:
# Variant 4: Redirect response

from django.shortcuts import get_object_or_404, redirect
from blog.models import Blog


def blog_detail(request, id):
    # get_object_or_404(Blog, id=id)
    return redirect('blog:blog_list')


# ❯ curl -i  "http://localhost:8000/blog/1/"
# HTTP/1.1 302 Found
# 302 == redirected request

🔑 After logging in, you can post it on the bulletin board.

After saving the comment, you can return it to the article.

In [ ]:
# Variant 5: Explicit 404 response

from django.http import Http404, HttpResponse
from blog.models import Blog


def blog_detail(request, id):
    # Keep explicit example of raising Http404 manually.
    try:
        blog = Blog.objects.get(id=id)
    except Blog.DoesNotExist as exc:
        raise Http404('Blog not found') from exc

    return HttpResponse(f"Blog found: {blog.title}")


# ❯ curl "http://localhost:8000/blog/1/"
# Blog found: Django Bootcamp%
#   <header id="summary">
#     <h1>Page not found <small>(404)</small></h1>
#     <pre class="exception_value">Blog not found</pre>

Or an explicit error message.

In [ ]:
# Create the template used by blog_list_example
# You need to create the template file in the templates directory.
# mysite/blog/templates/blog/blog_list_static_example.html
"""
<!doctype html>
<html>
  <head>
    <meta charset=\"utf-8\">
    <title>Blog List</title>
  </head>
  <body>
    <h1>Blog List</h1>
    <ul>
      {% for blog in blogs %}
        <li>
          <strong>{{ blog.title }}</strong> by {{ blog.author }}
        </li>
      {% empty %}
        <li>No blogs yet.</li>
      {% endfor %}
    </ul>
  </body>
</html>
"""

In [ ]:
# Example: Rendering templates (most common)

from django.shortcuts import render

def blog_detail(request: HttpRequest, id: int):
    # No need to use it in this example
    blogs = [
        {'id': 1, 'title': 'Django for Beginners', 'author': 'William Vincent'},
        {'id': 2, 'title': 'Two Scoops of Django', 'author': 'Daniel Feldroy'},
        {'id': 3, 'title': 'Django for Professionals', 'author': 'William Vincent'},
    ]
    return render(request, 'blog/blog_list_static_example.html', {'blogs': blogs})


# ❯ curl "http://localhost:8000/blog/1/"
# <!doctype html>
# <html>
#   <head>
#     <meta charset=\"utf-8\">
#     <title>Blog List</title>
#   </head>
#   <body>
#     <h1>Blog List</h1>
#     <ul>
#       ...

<br>

## Request and Response Objects

---

<div style="text-align: center;">
    <img src="../images/two-arrows.png" width="230" height="230">
</div>

### The Request Object

---

The `HttpRequest` object contains all information about the incoming request:

<br>

| Attribute | Description | Example |
|-----------|-------------|---------|
| `request.method` | HTTP method | `'GET'`, `'POST'` |
| `request.path` | URL path | `'/books/5/'` |
| `request.GET` | Query parameters | `{'page': '2'}` |
| `request.POST` | POST data | `{'title': 'New Book'}` |
| `request.user` | Current user | `<User: alice>` |
| `request.session` | Session data | `{'cart': [...]}` |
| `request.COOKIES` | Cookies | `{'csrftoken': '...'}` |
| `request.META` | HTTP headers | `{'HTTP_HOST': '...'}` |

In [ ]:
# Example: Using request attributes inside blog_detail

from http import HTTPStatus
from django.http import HttpRequest, JsonResponse
from models import Blog


def blog_detail(request: HttpRequest, id: int) -> JsonResponse:
    try:
        result = Blog.objects.get(id=id)

    except Blog.DoesNotExist:
        return JsonResponse({'error': 'Blog not found'},
                            status=HTTPStatus.NOT_FOUND)

    payload = {
        'id': result.id,
        'title': result.title,
        'author': result.author,
        'published_date': result.published_date.isoformat(),
    }

    if request.GET.get('debug') in {'1', 'true', 'yes'}:
        payload['debug'] = {
            'method': request.method,
            'path': request.path,
            'full_url': request.build_absolute_uri(),
            'query_params': dict(request.GET),
            'user': str(request.user),
            'is_authenticated': request.user.is_authenticated,
            'user_agent': request.META.get('HTTP_USER_AGENT', ''),
        }

    return JsonResponse(payload)


# ❯ curl "http://localhost:8000/blog/1/?debug=1"
# {"id": 1,
#  "title": "Django Bootcamp",
#  "author": "Matous Holinka",
#  "published_date": "2026-03-26",
#  "debug": {
#    "method": "GET",
#    "path": "/blog/1/",
#    "full_url": "http://localhost:8000/blog/1/?debug=1",
#    "query_params": {"debug": ["1"]},
#    "user": "AnonymousUser",
#    "is_authenticated": false,
#    "user_agent": "curl/8.7.1"
#  }}%  

🔑 Better choices:
- **Django Debug Toolbar:** The industry standard for local development. It provides a dedicated UI to inspect SQL queries, headers, and state without polluting your code.

- **Custom Middleware:** Better for global logic. It keeps your views DRY (Don't Repeat Yourself) by handling the ?debug=1 parameter in one place instead of modifying every single view.

- **Logging:** Use `logger.debug()`. It keeps your response payloads clean and allows you to monitor the system behavior in the console or log files without exposing internal data to the end-user.



# Example: Handling POST data - create a new blog

We can send data to this route via `POST` (form fields or JSON payload), not only read data via `GET`.

In [ ]:
import json

from django.http import JsonResponse
from blog.models import Blog
from django.views.decorators.csrf import csrf_exempt

@csrf_exempt  # Only for demo/curl testing - in production use CSRF tokens
def blog_create(request):
    # Demo-only example: keep it intentionally minimal.
    if request.method != 'POST':
        return JsonResponse({'tip': 'Send POST with title, author, published_date'})

    data = json.loads(request.body)
    blog = Blog.objects.create(
        title=data['title'],
        author=data['author'],
        published_date=data['published_date'],
    )
    return JsonResponse({'id': blog.id, 'title': blog.title}, status=201)

# Add a new URL route:
# mysite/blog/urls.py:
#   `path('create/', views.blog_create, name='blog_create')`

`@csrf_exempt` This decorator tells Django **to skip the security check for this specific view**.

It’s like leaving the back door unlocked so you can test your code easily with tools like `curl` without needing a security token.

<br>

`Happy-case` (`201 Created`):

```bash
❯ curl -X POST http://localhost:8000/blog/create/ \   
  -H "Content-Type: application/json" \
  -d '{"title": "My Blog", "author": "John", "published_date": "2024-01-01"}'
{"id": 2, "title": "My Blog"}%  
```

`Worst-case` (missing required key -> `500` in this minimal demo):

```bash
curl -X POST http://localhost:8000/blog/create/ \
  -H "Content-Type: application/json" \
  -d '{"title": "My Blog", "author": "John"}'
```

Note: In production, this should return `400 Bad Request` with validation errors instead of `500`.

🔑 **Advantages**: Complexity, capacity and cleanliness.

🔑 **Disadvantages**: non-shareable, overhead.

#### Request and query parameters 

---

Sometimes it's not enough to just read or send data.

Other times you'll need to **filter it**.

Typically for `GET requests`, when you just want to read, filter, or sort the data.

In [ ]:
# Example: Working with query parameters
from django.http import HttpRequest, JsonResponse
from models import Blog


def blog_search(request: HttpRequest) -> JsonResponse:    
    query = request.GET.get('q', '')          # 'django' or empty string
    sort = request.GET.get('sort', 'title')   # 'title' (default)
    blogs = Blog.objects.all()
    
    if query:
        blogs = blogs.filter(title__icontains=query)
    
    if sort in ('title', 'author', 'published_date'):
        blogs = blogs.order_by(sort)
    
    return JsonResponse(list(blogs.values()), safe=False)

# URL pattern: path('search/', views.blog_search, name='blog_search')
# Try in terminal:
# curl "http://localhost:8000/blog/search/?q=django&sort=published_date"
# [{"id": 1, "title": "Django Bootcamp", "author": "Matous Holinka", "published_date": "2026-03-26"}]% 

**Why these two checks matter**

- `blogs.filter(title__icontains=query)` runs only when `query` exists, so empty search does not add unnecessary filtering.
- `title__icontains` means: search in `title`, case-insensitive (`i`), and partial match (`contains`).
- `if sort in ('title', 'author', 'published_date')` is a small allowlist that prevents invalid or unsafe ordering input.
- `blogs.order_by(sort)` applies sorting only for known fields, keeping behavior predictable.

🔑 **Common pitfalls**
- allowlist for queries,
- convenient `icontains`,
- sorting.

TODO: Explain better case-sensitivy issue 

In [ ]:
# Example: Handling GET and POST in one view (realistic demo)

import json

from django.http import JsonResponse
from blog.models import Blog
from django.views.decorators.http import require_http_methods
from django.views.decorators.csrf import csrf_exempt

@csrf_exempt
@require_http_methods(["GET", "POST"])
def blog_list_create(request):
    if request.method == 'GET':
        # Real case: return latest blogs from the same endpoint.
        blogs = list(
            Blog.objects.order_by('-id').values('id', 'title', 'author', 'published_date')
        )
        return JsonResponse({'count': len(blogs), 'results': blogs})

    if request.method == 'POST':
        data = json.loads(request.body)
        blog = Blog.objects.create(
            title=data['title'],
            author=data['author'],
            published_date=data['published_date'],
        )
        return JsonResponse({'id': blog.id, 'title': blog.title}, status=201)

# GET demo (list latest blogs):
# curl "http://localhost:8000/blog/list-create/"
# {"count": 3,
#  "results": [
#    {"id": 3,
#     "title": "My Blog",
#     "author": "John",
#     "published_date": "2024-01-01"},
#    ...
#  ]}%
# POST demo (create blog):
# curl -X POST "http://localhost:8000/blog/list-create/" \
#   -H "Content-Type: application/json" \
#   -d '{"title":"Fast API Introduction",
#         "author":"Jane Doe",
#         "published_date":"2025-06-21"}'
# {"id": 6, "title": "Fast API Introduction"}% 

`@require_http_methods` This acts as a filter that only allows specific types of requests, like GET or POST. If someone tries to access the view using a different method, Django will automatically block them.


In [ ]:
from django.urls import path
from blog import views

app_name = 'blog'

urlpatterns = [
    path('', views.blog_list, name='blog_list'),
    path('search/', views.search_blogs, name='blog_search'),
    path('<int:id>/', views.blog_detail, name='blog_detail'),
    path('list-create/', views.blog_list_create, name='blog_list_create'),
]

<br>

### The Response Object

---

<div style="text-align: center;">
    <img src="../images/mail.png" width="230" height="230">
</div>

The `HttpResponse` object represents the response you send back:

<br>

| Method/Attribute | Description |
|------------------|-------------|
| `HttpResponse(content)` | Create response with content |
| `response.status_code` | HTTP status code (200, 404, etc.) |
| `response['Header']` | Set response headers |
| `response.set_cookie()` | Set a cookie |

In [ ]:
# Example: JSON response used in our views

from django.http import JsonResponse, HttpRequest
from http import HTTPStatus


def blog_detail_preview(request: HttpRequest) -> JsonResponse:
    """Return JSON payload similar to blog_detail()."""
    payload = {
        'status': 'ok',
        'method': request.method,
        'path': request.path,
    }

    if request.GET.get('debug') in {'1', 'true', 'yes'}:
        payload['debug'] = {
            'full_url': request.build_absolute_uri(),
            'query_params': dict(request.GET),
            'user_agent': request.META.get('HTTP_USER_AGENT', ''),
        }

    return JsonResponse(payload, status=HTTPStatus.OK)


# URL Route:
# mysite/blog/urls.py:
# path('preview/', views.blog_detail_preview, name='blog_detail_preview'),
# ❯ curl "http://localhost:8000/blog/preview/"              
# {"status": "ok", "method": "GET", "path": "/blog/preview/"}%   

In [ ]:
# Example: Response shortcuts we actually use

from http import HTTPStatus
from django.http import HttpResponse, JsonResponse


def blog_not_found_response(request: HttpRequest) -> JsonResponse:
    return JsonResponse(
        {'error': 'Blog not found'},
        status=HTTPStatus.NOT_FOUND,
    )


def healthcheck_response(request: HttpRequest) -> HttpResponse:
    return HttpResponse('OK', status=HTTPStatus.OK)


# URL Route:
# mysite/blog/urls.py:
# path('healthcheck/', views.healthcheck_response, name='healthcheck')
# ❯ curl "http://localhost:8000/blog/healthcheck/"
# OK%   

🔑 *Status codes* are like “traffic lights” on the internet.

Without them, communication between your server and the rest of the world (browser, mobile app, Google bot) would be one big mess.

[List of status codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status#informational_responses)
```
# Full overview of common HTTP status codes:
#
# 2xx - Success
#   200 OK              - standard successful response
#   201 Created         - resource was created (POST)
#   204 No Content      - success, no body (DELETE)
#
# 3xx - Redirection
#   301 Moved Permanently - URL changed forever
#   302 Found             - temporary redirect
#
# 4xx - Client errors
#   400 Bad Request     - invalid input from client
#   401 Unauthorized    - not authenticated
#   403 Forbidden       - authenticated but not allowed
#   404 Not Found       - resource doesn't exist
#   405 Method Not Allowed - wrong HTTP method
#
# 5xx - Server errors
#   500 Internal Server Error - unhandled exception in view
```

<br>

## Class-Based Views (CBV)

---

### Basic CBV Structure

---

A **class-based view** (CBV) is a Python class that handles HTTP requests:

- Inherits from `View` (or its subclasses)
- Defines methods for HTTP verbs: `get()`, `post()`, `put()`, `delete()`, etc.
- Uses `.as_view()` in URL configuration

In [ ]:
# Example: Basic class-based view

from django.http import HttpRequest, HttpResponse
from django.views import View

class HelloView(View):
    """A simple class-based view."""
    
    def get(self, request: HttpRequest) -> HttpResponse:
        """Handle GET requests."""
        return HttpResponse("Hello from CBV!")

In [ ]:
# Example: CBV with URL parameter

from django.http import HttpRequest, JsonResponse
from django.views import View
from django.shortcuts import get_object_or_404
from models import Blog


class BlogDetailView(View):
    def get(self, request: HttpRequest, pk: int) -> JsonResponse:
        blog = get_object_or_404(Blog, pk=pk)
        return JsonResponse({
            'id': blog.id,
            'title': blog.title,
            'author': blog.author,
            'published_date': blog.published_date.isoformat(),
        })

# In blog/urls.py:
# path('blogs/<int:pk>/', BlogDetailView.as_view(), name='blog_detail')
# ❯ curl "http://localhost:8000/blog/1/"
# {"id": 1,
#  "title": "Django Bootcamp",
#  "author": "Matous Holinka",
#  "published_date": "2026-03-26"}%  

<br>

### HTTP Method Handlers

---

CBVs handle different HTTP methods with separate methods:

| HTTP Method | CBV Method |
|-------------|------------|
| GET | `get(self, request, *args, **kwargs)` |
| POST | `post(self, request, *args, **kwargs)` |
| PUT | `put(self, request, *args, **kwargs)` |
| PATCH | `patch(self, request, *args, **kwargs)` |
| DELETE | `delete(self, request, *args, **kwargs)` |

In [ ]:
from django.utils.decorators import method_decorator
from django.views.decorators.csrf import csrf_exempt


@method_decorator(csrf_exempt, name='dispatch')
class BlogCreateView(View):
    def get(self, request: HttpRequest, pk: int) -> JsonResponse:
        blog = get_object_or_404(Blog, pk=pk)
        return JsonResponse({
            'id': blog.id,
            'title': blog.title,
            'author': blog.author,
            'published_date': blog.published_date.isoformat(),
        })

    def post(self, request: HttpRequest) -> JsonResponse:
        if request.content_type == 'application/json':
            data = json.loads(request.body)
        else:
            data = request.POST

        blog = Blog.objects.create(
            title=data.get('title', ''),
            author=data.get('author', ''),
            published_date=data.get('published_date', ''),
        )
        return JsonResponse({'id': blog.id, 'title': blog.title}, status=HTTPStatus.CREATED)

Class-Based Views (CBVs) work differently than regular functions.

You can't just put `@csrf_exempt` on a class because it's designed for functions.

<br>

🔑 That’s why we use `@method_decorator`.

In [ ]:
# Example: CBV with shared setup logic

from django.http import HttpRequest, HttpResponse
from django.views import View

class BlogView(View):
    
    # Class attribute - shared data
    blogs = {
        1: {'title': 'Django Basics', 'author': 'Matous'},
        2: {'title': 'Advanced Django', 'author': 'David'},
    }
    
    def setup(self, request: HttpRequest, *args, **kwargs):
        """Called before dispatch - initialize shared state."""
        super().setup(request, *args, **kwargs)
        # Any setup logic here
    
    def get(self, request: HttpRequest, pk: int) -> HttpResponse:
        ...
    
    def delete(self, request: HttpRequest, pk: int) -> HttpResponse:
        ...

🔑 Writing **shared logic in methods** in Class-Based Views (CBV) is absolutely essential in Django once your view starts doing more than just one simple thing.

It helps keep your code DRY (Don't Repeat Yourself) and clean.

In [ ]:
# Example: Using TemplateView for simple template rendering

from django.views.generic import TemplateView
from blog.models import Blog

class BlogListTemplateView(TemplateView):
    """Display list of blogs using a template."""
    template_name = 'blog/blog_list.html'

    def get_context_data(self, **kwargs):
        """Add blogs to the template context."""
        context = super().get_context_data(**kwargs)
        context['blogs'] = Blog.objects.all()
        context['title'] = 'All Blog Posts'
        return context

# Even simpler - directly in urls.py:
# path('blogs/', BlogListTemplateView.as_view(), name='blog_list_template')

In [ ]:
  # TemplateView přímo v urls.py (bez views.py)
    path('about/', TemplateView.as_view(template_name='blog/about.html'), name='about'),

<br>

## FBV vs CBV - When to Use Which

---

### 🔑 Comparison

---

| Aspect | Function-Based Views | Class-Based Views |
|--------|---------------------|-------------------|
| **Simplicity** | ✅ Easy to understand | ❌ More complex |
| **Explicit** | ✅ All logic visible | ❌ Hidden in inheritance |
| **Reusability** | ❌ Harder to reuse | ✅ Inheritance, mixins |
| **HTTP methods** | Manual if/else | Separate methods |
| **Decorators** | Simple to apply | Use `method_decorator` |
| **Testing** | Straightforward | May need more setup |

In [ ]:
# Comparison: Same functionality - FBV vs CBV

# === FBV approach ===
from django.http import HttpResponse
from django.views.decorators.http import require_http_methods

@require_http_methods(["GET", "POST"])
def blog_form_fbv(request, pk=None):
    if request.method == 'GET':
        # Show form
        return HttpResponse("Showing blog form")
    else:  # POST
        # Process form
        return HttpResponse("Processing blog form")

In [ ]:
# === CBV approach ===
from django.http import HttpResponse
from django.views import View

class BlogFormView(View):
    def get(self, request, pk=None):
        """Show the form."""
        return HttpResponse("Showing blog form")
    
    def post(self, request, pk=None):
        """Process the form."""
        return HttpResponse("Processing blog form")

<br>

### Practical Guidelines

---

**Use FBV when:**

- Simple, one-off views
- Learning Django (easier to understand)
- Views with complex, custom logic
- Single HTTP method handling

<br>

**Use CBV when:**

- CRUD operations (use generic views)
- Need to reuse code across views
- Multiple HTTP methods with shared setup
- Following RESTful patterns

In [ ]:
# Example: When FBV is cleaner

from django.http import JsonResponse

def health_check(request):
    """Simple health check endpoint."""
    return JsonResponse({'status': 'ok'})

# No need for a class here - it would be overkill

In [ ]:
# Example: When CBV shines - reusable base class

from django.views import View
from django.http import JsonResponse

class APIView(View):
    """Base class for API views."""
    
    def dispatch(self, request, *args, **kwargs):
        """Add common API logic."""
        # Check API key, rate limiting, etc.
        return super().dispatch(request, *args, **kwargs)
    
    def json_response(self, data, status=200):
        """Helper method for JSON responses."""
        return JsonResponse(data, status=status)


class BookAPIView(APIView):
    """Book API - inherits common API logic."""
    
    def get(self, request, pk=None):
        if pk:
            return self.json_response({'id': pk, 'title': 'Book'})
        return self.json_response({'books': []})


class AuthorAPIView(APIView):
    """Author API - also inherits common API logic."""
    
    def get(self, request, pk=None):
        return self.json_response({'authors': []})

<br>

## Useful View Decorators and Shortcuts

---

In [ ]:
# Common decorators for FBVs

from django.views.decorators.http import require_GET, require_POST, require_http_methods
from django.views.decorators.csrf import csrf_exempt
from django.contrib.auth.decorators import login_required

@require_GET
def list_view(request):
    """Only accepts GET requests."""
    return HttpResponse("List")

@require_POST
def create_view(request):
    """Only accepts POST requests."""
    return HttpResponse("Created")

@require_http_methods(["GET", "POST"])
def form_view(request):
    """Accepts GET and POST only."""
    return HttpResponse("Form")

@login_required
def protected_view(request):
    """Requires user to be logged in."""
    return HttpResponse(f"Hello, {request.user}!")

In [ ]:
# Common shortcuts

from django.shortcuts import render, redirect, get_object_or_404, get_list_or_404

def book_detail(request, pk):
    """Using get_object_or_404."""
    # Raises Http404 if not found
    # book = get_object_or_404(Book, pk=pk)
    # return render(request, 'book_detail.html', {'book': book})
    pass

def redirect_example(request):
    """Using redirect shortcut."""
    # Redirect to named URL
    return redirect('catalog:book_list')
    
    # Or to a URL path
    # return redirect('/books/')
    
    # Or to an object with get_absolute_url()
    # return redirect(book)

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **FBV** | Simple function: `def view(request)` → `HttpResponse` |
| **CBV** | Class with methods: `get()`, `post()`, etc. |
| **Request object** | `request.method`, `.GET`, `.POST`, `.user` |
| **Response object** | `HttpResponse`, `JsonResponse`, `redirect()` |
| **render()** | Shortcut to render template with context |
| **URL parameters** | Passed as function/method arguments |
| **When FBV** | Simple views, custom logic, learning |
| **When CBV** | CRUD, reusability, multiple HTTP methods |

<br>

### 🧠 EXERCISE 🧠, Create Views for Bookstore

---

Using the `mysite` project:

1. Create a FBV `blog_list` that returns a list of books as JSON
2. Create a FBV `blog_detail` that takes `pk` and returns book info or 404
3. Create a CBV `BlogSearchView` that:
   - GET: returns search form (just text for now)
   - POST: returns search results based on `request.POST.get('query')`
4. Add URL patterns for all views
5. Test all endpoints with the browser and/or curl

<details>
    <summary>▶️ Solution</summary>

**1. Add to `blog/views.py`:**

```python
from django.http import JsonResponse, HttpRequest
from django.views import View
from django.shortcuts import get_object_or_404
from blog.models import Blog
from http import HTTPStatus


# FBV: blog_list
def blog_list(request: HttpRequest) -> JsonResponse:
    """Return list of all blogs as JSON."""
    blogs = list(
        Blog.objects.all().values('id', 'title', 'author', 'published_date')
    )
    return JsonResponse({'count': len(blogs), 'results': blogs})


# FBV: blog_detail
def blog_detail(request: HttpRequest, pk: int) -> JsonResponse:
    """Return single blog by pk or 404."""
    blog = get_object_or_404(Blog, pk=pk)
    return JsonResponse({
        'id': blog.id,
        'title': blog.title,
        'author': blog.author,
        'published_date': blog.published_date.isoformat(),
    })


# CBV: BlogSearchView
class BlogSearchView(View):
    def get(self, request: HttpRequest) -> JsonResponse:
        """Return search form info."""
        return JsonResponse({
            'message': 'Search form',
            'hint': 'POST with {"query": "search term"}'
        })
    
    def post(self, request: HttpRequest) -> JsonResponse:
        """Return search results based on query."""
        import json
        data = json.loads(request.body)
        query = data.get('query', '')
        
        blogs = Blog.objects.filter(title__icontains=query)
        results = list(blogs.values('id', 'title', 'author'))
        
        return JsonResponse({
            'query': query,
            'count': len(results),
            'results': results
        })
```

**2. Add to `blog/urls.py`:**

```python
from django.urls import path
from blog import views

urlpatterns = [
    path('blogs/', views.blog_list, name='blog_list'),
    path('blogs/<int:pk>/', views.blog_detail, name='blog_detail'),
    path('search/', views.BlogSearchView.as_view(), name='blog_search'),
]
```

**3. Test:**

```bash
# Test blog_list
curl http://127.0.0.1:8000/blog/blogs/

# Test blog_detail
curl http://127.0.0.1:8000/blog/blogs/1/

# Test search (GET)
curl http://127.0.0.1:8000/blog/search/

# Test search (POST)
curl -X POST http://127.0.0.1:8000/blog/search/ \
  -H "Content-Type: application/json" \
  -d '{"query": "Django"}'
```
</details>

<br>

### 🧠 EXERCISE 🧠, Add Request Information View

---

Create a debug view that displays request information:

1. Create a FBV `debug_request` that returns:
   - HTTP method
   - Path
   - Query parameters (if any)
   - User (authenticated or anonymous)
   - User-Agent header
2. Add URL pattern at `/debug/`
3. Test with: `http://127.0.0.1:8000/debug/?foo=bar&page=2`

<details>
    <summary>▶️ Solution</summary>

**1. Add to `blog/views.py`:**

```python
def debug_request(request):
    """Display debug information about the request."""
    info = {
        'method': request.method,
        'path': request.path,
        'query_params': dict(request.GET),
        'user': str(request.user),
        'is_authenticated': request.user.is_authenticated,
        'user_agent': request.META.get('HTTP_USER_AGENT', 'Unknown'),
    }
    return JsonResponse(info, json_dumps_params={'indent': 2})
```

**2. Add to `blog/urls.py`:**

```python
urlpatterns = [
    # ... existing patterns ...
    path('debug/', views.debug_request, name='debug'),
]
```

**3. Test:**

```bash
curl "http://127.0.0.1:8000/blog/debug/?foo=bar&page=2"
```

Expected output:
```json
{
  "method": "GET",
  "path": "/blog/debug/",
  "query_params": {"foo": ["bar"], "page": ["2"]},
  "user": "AnonymousUser",
  "is_authenticated": false,
  "user_agent": "curl/7.68.0"
}
```
</details>

---